In [7]:
from transformers import GPT2TokenizerFast
from transformers import AutoModelForCausalLM
import pickle
import numpy as np
# Initialize tokenizer and model
model_name ="DeepESP/gpt2-spanish"
#n_features = 768

tokenizer = GPT2TokenizerFast.from_pretrained(model_name, add_prefix_space=True, cache_dir = "/data/u_barchet_software/cache/")
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = "/data/u_barchet_software/cache/")

# Print out model architecture
model.eval()

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 26459.68it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
import pandas as pd

events_meg = pd.read_csv("../brain2qwerty/SpanishBCBL/Pinet2024Meg/events.csv")

In [31]:
events_perception = events_meg[(events_meg['is_percep'] == True) & (events_meg['type'] == "Word")]
events_perception = events_perception[['type', 'start', 'stop', 'subject', 'session', 'task', 'text', 'trial_id', 'true_sequence']]

events_perception = events_perception[events_perception['trial_id'] > 3]

In [32]:
sentences_uni = events_perception['true_sequence'].unique()
sentences_ids = pd.DataFrame(zip(sentences_uni, range(len(sentences_uni))), columns = ['true_sequence', 'sid'])

events_perception_id = pd.merge(events_perception, sentences_ids, on = "true_sequence")

In [ ]:
unique_df = events_perception_id[['sid', 'text']].drop_duplicates().reset_index(drop=True)

In [ ]:
import pandas as pd
from textgrid import TextGrid
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator

from os import listdir
from os.path import isfile, join


ids = unique_df


emb_all = []
for sid in unique_df['sid'].unique():

    df = unique_df[unique_df['sid'] == sid]
    
    
    
    df['word_number'] = range(1, len(df) + 1)

    
    # Initialize tokenizer and model
    model_name = "DeepESP/gpt2-spanish"
    
    device = "cpu"
    model.to(device)
    
    
    
    # Reload in transcript CSV file
    transcript_gpt2 = df.copy()
    
    # Insert explicit index column for reference
    transcript_gpt2.insert(0, 'word_index', transcript_gpt2.index.values)
    
    # Tokenize words into lists of tokens
    transcript_gpt2['token'] = transcript_gpt2.text.apply(tokenizer.tokenize)
    
    # "Explode" lists of token subwords into long format
    transcript_gpt2 = transcript_gpt2.explode('token', ignore_index=True)
    
    # Convert tokens to token IDs for input to model
    transcript_gpt2['token_id'] = transcript_gpt2.token.apply(tokenizer.convert_tokens_to_ids)


    
    
    # Convert all token IDs into list
    token_ids = transcript_gpt2.token_id.tolist()
    
    # Extract context window width for model
    max_len = 100
    
    # Compile into lists of tokens within each context window
    samples = []
    token_ids = torch.tensor(transcript_gpt2.token_id.tolist(), dtype=torch.long)
    samples.append(token_ids[0:max_len])
    for i in range(max_len+1, len(token_ids)+1):
        samples.append(token_ids[i-max_len:i])
    

    # Initialize accelerator and free memory
    accelerator = Accelerator()
    accelerator.free_memory()
    
    # Send model to device
    model = model.to(device)
    

    # Set a batch size for the data loader
    batch_size = 4
    
    # Extract a late-intermediate layer from GPT-2
    layer = 8
    
    # Extract embeddings and other model features
    embeddings = []
    top_guesses = []
    ranks = []
    true_probs = []
    entropies = []
    with torch.no_grad():
        data_loader = torch.utils.data.DataLoader(samples, batch_size=batch_size,
                                                  shuffle=False)
    
        # Loop through samples and extract embeddings
        for i, batch in enumerate(tqdm(data_loader)):
            output = model(batch.to(device), output_hidden_states=True)
            logits = output.logits  # torch.Size([2, 1024, 50257])
            states = output.hidden_states[layer]
    
            # Extract all embeddings/features for first context window
            if i == 0:
                true_ids = batch[0, :]
                brange = list(range(len(true_ids)-1))
                logits_order = logits[0].argsort(descending=True, dim=-1)
                batch_top_guesses = logits_order[:-1, 0]
                print(batch_top_guesses)
                batch_ranks = torch.eq(logits_order[:-1],
                                       true_ids.reshape(-1,1)[1:].to(device)).nonzero()[:, 1]
                batch_probs = logits[0, :-1].softmax(-1)
                batch_true_probs = batch_probs[brange, true_ids[1:]]

                batch_entropy = torch.distributions.Categorical(
                    logits=logits[0, :-1].float()
                ).entropy()

               # batch_entropy = torch.distributions.Categorical(probs=batch_probs).entropy()
                batch_embeddings = states[0]
    
                top_guesses.append(batch_top_guesses.numpy(force=True))
                ranks.append(batch_ranks.numpy(force=True))
                true_probs.append(batch_true_probs.numpy(force=True))
                entropies.append(batch_entropy.numpy(force=True))
                embeddings.append(batch_embeddings.numpy(force=True))
                
                # Reset if there are samples remaining in this batch
                if batch.size(0) == 1:
                    continue
                logits = logits[1:]
                states = states[1:]
                batch = batch[1:]
    
            # Extract embeddings/features for last word in subsequent windows
            true_ids = batch[:, -1]
            brange = list(range(len(true_ids)))
            logits_order = logits[:, -2, :].argsort(descending=True)  # batch x vocab_size
            batch_top_guesses = logits_order[:, 0]
            batch_ranks = torch.eq(logits_order, true_ids.reshape(-1,1).to(device)).nonzero()[:, 1]
            batch_probs = torch.softmax(logits[:, -2, :], dim=-1)
            batch_true_probs = batch_probs[brange, true_ids]
            batch_entropy = torch.distributions.Categorical(probs=batch_probs).entropy()
            batch_embeddings = states[:, -1, :]

            print(batch_top_guesses)

            top_guesses.append(batch_top_guesses.numpy(force=True))
            ranks.append(batch_ranks.numpy(force=True))
            true_probs.append(batch_true_probs.numpy(force=True))
            entropies.append(batch_entropy.numpy(force=True))
            embeddings.append(batch_embeddings.numpy(force=True))

                # Compile outputs into transcript (logit derivatives must be shifted by 1)
    transcript_gpt2.loc[1:, 'rank'] = np.concatenate(ranks)
    transcript_gpt2.loc[1:, 'true_prob'] = np.concatenate(true_probs)
    transcript_gpt2.loc[1:, 'top_pred'] = np.concatenate(top_guesses)
    transcript_gpt2.loc[0, 'top_pred'] = tokenizer.bos_token_id
    transcript_gpt2.loc[1:, 'entropy'] = np.concatenate(entropies)
    transcript_gpt2['embedding'] = [e for e in np.vstack(embeddings)]
    
    # Reduce size of transcript
    transcript_gpt2 = transcript_gpt2.astype({'word_index': 'int32', 'token_id': 'int32',
                                          'rank': 'float32', 'true_prob': 'float32',
                                          'top_pred': 'int32', 'entropy': 'float32', 
                                             }, copy=False)
    
    # Convert model's top predictions from token IDs to tokens
    transcript_gpt2['top_pred'] = transcript_gpt2.top_pred.apply(tokenizer.convert_ids_to_tokens)
    
    # Print out top-1 and top-10 word prediction accuracy
    print(f"Top-1 accuracy: {(transcript_gpt2['rank'] == 0).mean():.3f}")
    print(f"Top-10 accuracy: {(transcript_gpt2['rank'] < 10).mean():.3f}")
    
    mean_embeddings = (
        transcript_gpt2.groupby('word_number', as_index=False)
          .agg({
              'embedding': lambda x: np.mean(np.stack(x), axis=0),
              'text': 'first',
            'word_index': 'first',
    
              
              'sid':'first'
          })
        )

    emb_all.append(mean_embeddings)



  0%|          | 0/1 [00:00<?, ?it/s]

tensor([1238,  329,  268, 4505,  329,   23,  288, 1817,  393])


100%|██████████| 1/1 [00:03<00:00,  3.14s/it]
/tmp/ipykernel_27783/2041811967.py:156: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  transcript_gpt2 = transcript_gpt2.astype({'word_index': 'int32', 'token_id': 'int32',


Top-1 accuracy: 0.100
Top-10 accuracy: 0.200


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

tensor([ 1238,   268,  1826,   335, 24685])


100%|██████████| 1/1 [00:02<00:00,  2.16s/it]

Top-1 accuracy: 0.000
Top-10 accuracy: 0.333



/tmp/ipykernel_27783/2041811967.py:156: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  transcript_gpt2 = transcript_gpt2.astype({'word_index': 'int32', 'token_id': 'int32',


In [ ]:
with open("./embeddings/embeddings_gpt.pickle", "wb") as o:
    pickle.dump(emb_all, o)